## Setup

In [7]:
from jaxcmr._analyses import _map_nonzero_recall_to_all_positions, map_recall_to_all_positions, map_single_trial_to_positions, item_occurrences, crp
from jaxcmr.datasets import load_data, generate_trial_mask
from jaxcmr.analyses import (
    single_pres_crp,
    multi_pres_crp,
)
import pytest
import numpy as np
from jax import numpy as jnp, lax, vmap

In [4]:
data_path = "D:/data/{}.h5"
param_path = "D:/data/results/{}_{}_{}.jsonl"

data_tag = "LohnasKahana2014"
trial_query = "data['list_type'] == 1"

data = load_data(data_path.format(data_tag))
trial_mask = generate_trial_mask(data, trial_query)

recalls = data["recalls"][trial_mask]
pres_itemnos = data["pres_itemnos"][trial_mask]

## Single Case

In [1]:
import jax
import jax.numpy as jnp
from jax import vmap, lax

def update_lags(previous_item, recall_item, lag_totals):
    actual_lags, possible_lags = lag_totals
    lag_range = jnp.size(actual_lags) // 2
    
    current_lag = recall_item - previous_item + lag_range
    actual_lags = jax.ops.index_add(actual_lags, jax.ops.index[current_lag], 1)
    
    return previous_item, (actual_lags, possible_lags)

def process_trial(trial_recalls, list_length):
    lag_range = list_length - 1
    actual_lags = jnp.zeros(lag_range * 2 + 1)
    possible_lags = jnp.zeros(lag_range * 2 + 1)
    
    terminus = jnp.sum(trial_recalls != 0)
    non_zero_recalls = trial_recalls[:terminus]

    possible_items = jnp.arange(1, list_length + 1)
    initial_item = jnp.array(0)
    
    # Process each recall in the current trial
    _, (actual_lags, possible_lags) = lax.scan(
        update_lags, 
        initial_item, 
        non_zero_recalls, 
        (actual_lags, possible_lags)
    )
    
    # Compute possible lags based on remaining items and last recalled item
    last_item = non_zero_recalls[-1] if terminus > 0 else 0
    remaining_possible_items = jnp.setdiff1d(possible_items, non_zero_recalls, assume_unique=True)
    remaining_lags = remaining_possible_items - last_item + lag_range
    updated_possible_lags = jax.ops.index_add(possible_lags, jax.ops.index[remaining_lags], 1)
    
    return actual_lags, updated_possible_lags

def single_pres_crp(recalls, list_length):
    lag_range = list_length - 1
    zero_lags = jnp.zeros((recalls.shape[0], lag_range * 2 + 1))
    
    # Vectorize process_trial over all trials
    actual_lags, possible_lags = vmap(process_trial, in_axes=(0, None))(recalls, list_length)
    
    # Sum over trials
    total_actual_lags = jnp.sum(actual_lags, axis=0)
    total_possible_lags = jnp.sum(possible_lags, axis=0)
    
    # Avoid division by zero
    safe_total_possible_lags = jax.ops.index_add(total_possible_lags, jax.ops.index[total_actual_lags == 0], 1)
    
    return total_actual_lags / safe_total_possible_lags

# Test the function
recalls = jnp.array([[1, 2, 0], [2, 1, 0], [3, 1, 2]])
list_length = 3
result = single_pres_crp(recalls, list_length)


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


IndexError: Array slice indices must have static start/stop/step to be used with NumPy indexing syntax. Found slice(None, Traced<ShapedArray(int32[])>with<BatchTrace(level=1/0)> with
  val = Array([2, 2, 3], dtype=int32)
  batch_dim = 0, None). To index a statically sized array at a dynamic position, try lax.dynamic_slice/dynamic_update_slice (JAX does not support dynamically sized arrays within JIT compiled functions).